<a href="https://colab.research.google.com/github/ArteM1421/FREE-UNCENSURED-AI/blob/main/Pony_Diffusion_V6_XL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## FREE UNCENSURED AI - **Pony diffusion V6 XL** (PHOTO)
Author GitHub - [**ArteM1421**](https://github.com/ArteM1421)  
GitHub REPO - [**CLICK**](https://github.com/ArteM1421/FREE-UNCENSURED-AI)


⚠️ **I am not responsible for what you have done in this tool. All responsibility lies with you.**

In [7]:
#@title **🚀 STEP 1: INSTALL REQUIREMENTS**
import subprocess, sys, platform, re

# ----------------------------- UI HELPERS -----------------------------
def progress_bar(percent, width=30, prefix=""):
    percent = max(0, min(100, percent))
    filled = int(width * percent / 100)
    bar = "█" * filled + "░" * (width - filled)
    return f"{prefix}[{bar}] {percent:5.1f}%"

def pip_install(pkgs, index_url=None, label=""):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs]
    if index_url:
        cmd += ["--index-url", index_url]
    return subprocess.run(cmd, capture_output=True, text=True)


def run_install(pkgs, index_url=None, label=""):
    total = len(pkgs)
    failed = []
    for i, pkg in enumerate(pkgs, start=1):
        pct = (i - 1) / total * 100
        print("\r" + progress_bar(pct, prefix=f"  {label} "), end="", flush=True)
        res = pip_install([pkg], index_url=index_url)
        if res.returncode != 0:
            failed.append((pkg, res.stderr or ""))
    print("\r" + progress_bar(100, prefix=f"  {label} "))
    return failed


# ----------------------------- BASE PACKAGES --------------------------
packages = [
    ("diffusers",    "🎨"),
    ("transformers", "🤗"),
    ("accelerate",   "⚡"),
    ("ipython",      "💻"),
    ("safetensors",  "🔒"),
]

print("⏳ Installing base requirements...\n")
failed = run_install(
    [p for p, _ in packages],
    label="base",
)

if failed:
    print("\n❌ Installation failed for:")
    for pkg, err in failed:
        print(f"\n--- {pkg} ---")
        err_lines = [
            line for line in err.splitlines()
            if line.strip() and "Requirement already satisfied" not in line
        ]
        for line in err_lines[-10:]:
            print(line)
    raise RuntimeError("pip install failed")
print("✅ Base packages installed.")

# ----------------------------- TORCH STACK ----------------------------
py_ver = sys.version_info[:2]
print(f"\n🐍 Python {platform.python_version()}")

if py_ver >= (3, 13):
    torch_pkgs = ["torch", "torchvision", "torchaudio"]
    index_url = "https://download.pytorch.org/whl/cu121"
else:
    torch_pkgs = ["torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1"]
    index_url = "https://download.pytorch.org/whl/cu121"

print(f"🔥 Installing torch stack: {' '.join(torch_pkgs)}\n")
failed_torch = run_install(torch_pkgs, index_url=index_url, label="torch")

if failed_torch:
    print("\n❌ Torch install failed:")
    for pkg, err in failed_torch:
        for line in (err or "").splitlines()[-15:]:
            print(line)
    raise RuntimeError("torch install failed")

import torch, torchvision
print(f"✅ torch {torch.__version__} | torchvision {torchvision.__version__}")
print("\n✅ Installation completed successfully.")

⏳ Installing base requirements...

  base [██████████████████████████████] 100.0%
✅ Base packages installed.

🐍 Python 3.13.16
🔥 Installing torch stack: torch torchvision torchaudio

  torch [██████████████████████████████] 100.0%

❌ Torch install failed:
ERROR: Could not find a version that satisfies the requirement torchaudio (from versions: none)
ERROR: No matching distribution found for torchaudio


RuntimeError: torch install failed

In [ ]:
#@title **🚀 STEP 2: FREE UNCENSORED AI — GENERATOR**
import os
import gc
import time
from datetime import datetime
from pathlib import Path

import torch
from diffusers import StableDiffusionXLPipeline
from IPython.display import display, clear_output

# ----------------------------- CONFIG -----------------------------
MODEL_ID   = "stablediffusionapi/Pony-Diffusion-V6-XL"
OUTPUT_DIR = Path("generated")
OUTPUT_DIR.mkdir(exist_ok=True)

NEGATIVE_PROMPT = "lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality, normal quality, jpeg artifacts, signature, watermark, username, blurry"

# ----------------------------- MENU -------------------------------
MENU = r"""
╔══════════════════════════════════════════════════════════════╗
║                                                              ║
║        ███████╗██████╗ ███████╗███████╗                       ║
║        ██╔════╝██╔══██╗██╔════╝██╔════╝                       ║
║        █████╗  ██████╔╝█████╗  █████╗                         ║
║        ██╔══╝  ██╔══██╗██╔══╝  ██╔══╝                         ║
║        ██║     ██║  ██║███████╗███████╗                       ║
║        ╚═╝     ╚═╝  ╚═╝╚══════╝╚══════╝                       ║
║                                                              ║
║           U N C E N S O R E D   A I                          ║
║                                                              ║
║                  GITHUB: ARTEM1421                           ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝

  [1]  Generate an image
  [2]  Generate multiple images
  [3]  Settings
  [0]  Exit
"""

# ----------------------------- SETTINGS ---------------------------
settings = {
    "steps": 30,
    "guidance": 7.0,
    "width": 832,
    "height": 1216,
    "seed": -1,
}

SIZE_PRESETS = [
    (512,  512,  "tiny / fast"),
    (640,  640,  "small"),
    (768,  768,  "medium square"),
    (832,  1216, "SDXL portrait (recommended)"),
    (1216, 832,  "SDXL landscape"),
    (1024, 1024, "SDXL square"),
    (1024, 1536, "large portrait"),
    (1536, 1024, "large landscape"),
]

# ----------------------------- PIPELINE ---------------------------
_pipe = None

def get_pipe():
    global _pipe
    if _pipe is None:
        print("\n[!] Loading model, please wait (first time ~1-2 min)...\n")
        _pipe = StableDiffusionXLPipeline.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16,
            variant="fp16",
        )
        _pipe.enable_model_cpu_offload()
        try:
            _pipe.enable_vae_slicing()
            _pipe.enable_vae_tiling()
        except Exception:
            pass
        _pipe.set_progress_bar_config(disable=False)
        print("\n[✓] Model loaded.\n")
    return _pipe

# ----------------------------- HELPERS ----------------------------
def estimate_time(w, h, steps):
    base_pixels = 832 * 1216
    base_seconds = 27.0
    return base_seconds * ((w * h) / base_pixels) * (steps / 30.0)

def estimate_memory(w, h):
    base_pixels = 832 * 1216
    return 10.5 * ((w * h) / base_pixels)

def safe_resolution(w, h, mem_limit_gb=14.0):
    w, h = int(w), int(h)
    while estimate_memory(w, h) > mem_limit_gb and w > 256 and h > 256:
        w = int(w * 0.9)
        h = int(h * 0.9)
    w = max(256, (w // 8) * 8)
    h = max(256, (h // 8) * 8)
    return w, h

def free_memory():
    gc.collect()
    torch.cuda.empty_cache()

# ----------------------------- GENERATE ---------------------------
def generate(prompt: str, n: int = 1):
    pipe = get_pipe()

    w, h = settings["width"], settings["height"]
    est_mem = estimate_memory(w, h)
    est_time = estimate_time(w, h, settings["steps"])

    if est_mem > 14.0:
        safe_w, safe_h = safe_resolution(w, h)
        print(f"\n[!] {w}x{h} likely too big for 16GB GPU "
              f"(~{est_mem:.1f} GB needed).")
        print(f"[!] Auto-reducing to {safe_w}x{safe_h}.")
        w, h = safe_w, safe_h
        est_mem = estimate_memory(w, h)
        est_time = estimate_time(w, h, settings["steps"])

    seed = settings["seed"]
    if seed == -1:
        seed = torch.randint(0, 2**32 - 1, (1,)).item()
    gen = torch.Generator(device="cuda").manual_seed(seed)

    print(f"\n» Prompt : {prompt}")
    print(f"» Seed   : {seed}")
    print(f"» Size   : {w}x{h}  (~{est_mem:.1f} GB VRAM, ~{est_time:.0f}s)")
    print(f"» Steps  : {settings['steps']}  |  CFG: {settings['guidance']}\n")

    free_memory()
    start = time.time()

    try:
        result = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=settings["steps"],
            guidance_scale=settings["guidance"],
            width=w,
            height=h,
            generator=gen,
            num_images_per_prompt=n,
        )
    except torch.cuda.OutOfMemoryError:
        free_memory()
        w2, h2 = max(256, (w // 2) // 8 * 8), max(256, (h // 2) // 8 * 8)
        print(f"[!] OOM caught. Retrying at {w2}x{h2}...")
        result = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=settings["steps"],
            guidance_scale=settings["guidance"],
            width=w2,
            height=h2,
            generator=gen,
            num_images_per_prompt=n,
        )
        w, h = w2, h2

    elapsed = time.time() - start

    saved = []
    for i, img in enumerate(result.images):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        path = OUTPUT_DIR / f"{ts}_{i}_{w}x{h}_seed{seed}.png"
        img.save(path)
        saved.append(path)
        display(img)

    print(f"\n[✓] Done in {elapsed:.1f}s. Saved to: {OUTPUT_DIR}/")
    for p in saved:
        print(f"    • {p.name}")

    free_memory()

# ----------------------------- DISPLAY ----------------------------
def print_menu():
    clear_output(wait=True)
    print(MENU)
    w, h = settings["width"], settings["height"]
    est_mem = estimate_memory(w, h)
    est_time = estimate_time(w, h, settings["steps"])
    print(f"  Current settings: {w}x{h}, {settings['steps']} steps, "
          f"CFG {settings['guidance']}, seed {settings['seed']}")
    print(f"  Estimate: ~{est_mem:.1f} GB VRAM, ~{est_time:.0f}s per image\n")

def settings_menu():
    print("\n--- SETTINGS ---")
    print("\nSize presets:")
    for i, (pw, ph, label) in enumerate(SIZE_PRESETS, start=1):
        em = estimate_memory(pw, ph)
        et = estimate_time(pw, ph, settings["steps"])
        print(f"  [{i}] {pw}x{ph:<5}  {label:<30}  ~{em:4.1f} GB, ~{et:.0f}s")

    try:
        print()
        s = input(f"Steps [{settings['steps']}]: ").strip()
        if s: settings["steps"] = max(1, int(s))

        g = input(f"CFG [{settings['guidance']}]: ").strip()
        if g: settings["guidance"] = float(g)

        print("\nSize — enter preset number or custom WxH (e.g. 1024x1024)")
        sz = input(f"Size [{settings['width']}x{settings['height']}]: ").strip().lower()
        if sz:
            if sz.isdigit() and 1 <= int(sz) <= len(SIZE_PRESETS):
                pw, ph, _ = SIZE_PRESETS[int(sz) - 1]
                settings["width"], settings["height"] = pw, ph
            elif "x" in sz:
                try:
                    pw, ph = sz.split("x")
                    pw, ph = int(pw), int(ph)
                    if pw < 256 or ph < 256:
                        print("[!] Minimum size is 256x256.")
                    elif pw > 2048 or ph > 2048:
                        print("[!] Maximum size is 2048x2048.")
                    else:
                        settings["width"], settings["height"] = pw, ph
                except ValueError:
                    print("[!] Invalid size format.")
            else:
                print("[!] Invalid size input.")

        sd = input(f"Seed (-1=random) [{settings['seed']}]: ").strip()
        if sd: settings["seed"] = int(sd)
    except ValueError:
        print("[!] Invalid value, settings unchanged.")
        return
    print("[✓] Settings updated.")

# ----------------------------- MAIN LOOP --------------------------
def main():
    while True:
        print_menu()
        choice = input("Choice: ").strip()

        if choice == "0":
            print("\nGoodbye! 👋")
            break

        elif choice == "1":
            prompt = input("\nEnter prompt: ").strip()
            if not prompt:
                print("[!] Empty prompt.")
                continue
            try:
                generate(prompt, n=1)
            except Exception as e:
                print(f"[✗] Error: {e}")
                free_memory()

        elif choice == "2":
            prompt = input("\nEnter prompt: ").strip()
            if not prompt:
                print("[!] Empty prompt.")
                continue
            try:
                n = int(input("How many images? [4]: ").strip() or 4)
                generate(prompt, n=n)
            except Exception as e:
                print(f"[✗] Error: {e}")
                free_memory()

        elif choice == "3":
            settings_menu()

        else:
            print("[!] Unknown command.")

        input("\nPress Enter to continue...")


if __name__ == "__main__":
    main()